<a href="https://colab.research.google.com/github/jamagiwa/L-Trail/blob/main/tutorials/ltrail_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#L-Trail


Installation & Setup

In [ ]:
!git clone https://github.com/jamagiwa/L-Trail.git
%cd L-Trail
!pip install -r requirements.txt

In [ ]:
import scvelo as scv
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import scanpy as sc
from scipy.stats import gaussian_kde, skew
import random
from scipy.stats import skew
from sklearn.neighbors import NearestNeighbors
import seaborn as sns
from scipy.spatial.distance import cosine
from ltrail import pl
from ltrail import tl

In [ ]:
SEED = 34

random.seed(SEED)
np.random.seed(SEED)
sc.settings.seed = SEED
scv.settings.seed = SEED

In [ ]:
#Font settings for publication
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans', 'Bitstream Vera Sans']

# Figure size and font size settings
plt.rcParams['font.size'] = 8
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['axes.labelsize'] = 9
plt.rcParams['xtick.labelsize'] = 8
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['legend.fontsize'] = 8
plt.rcParams['figure.titlesize'] = 12

# Scanpy plotting configurations
sc.set_figure_params(
    scanpy=True,
    dpi=100,
    dpi_save=300,
    frameon=False,
    vector_friendly=True,
    fontsize=10,
    figsize=(6, 6),
    color_map='viridis'
    )

##Data Loading


In [ ]:
adata = scv.datasets.pancreas()

## RNA velocity

Process the dataset following the standard scVelo preprocessing and velocity estimation pipeline.

In [ ]:
scv.settings.verbosity = 3
scv.settings.presenter_view = True
scv.set_figure_params('scvelo')

scv.pp.filter_genes(adata,
                    min_shared_counts=20)
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
sc.pp.filter_genes_dispersion(adata, n_top_genes=2000)

scv.pp.moments(adata, n_pcs=30, n_neighbors=30)
scv.tl.velocity(adata, mode='stochastic')
scv.tl.velocity_graph(adata)

# Explicitly compute PCA and UMAP embeddings
sc.tl.pca(adata, svd_solver='arpack')
sc.tl.umap(adata)

In [ ]:
scv.tl.velocity_graph(adata)

In [ ]:
scv.pl.velocity_embedding_stream(adata,
                                 basis='umap',
                                 color='clusters',
                                 figsize=(8, 6),
                                 legend_loc='right margin',
                                 title='Pancreas RNA Velocity (Stochastic, umap)',
                                 show=False,
                                 )

scv.pl.velocity_embedding_stream(adata,
                                 basis='pca',
                                 color='clusters',
                                 figsize=(8, 6),
                                 legend_loc='right margin',
                                 title='Pancreas RNA Velocity (Stochastic, pca)',
                                 show=False
                                 )



##Run L-Trail

In [ ]:
#L-Trail on pca
pl.plot_ltrail(adata,
            groupby='clusters',
            basis='X_pca',
            use_rep='X_pca',
            method='lmoment',
            scale=30,
            p_threshold=0.05,
            title='Pancreas L-Trail(scale=30, p<0.05, pca)'
            )

#L-Trail on umap
pl.plot_ltrail(adata,
            groupby='clusters',
            basis='X_umap',
            use_rep='X_pca',
            method='lmoment',
            scale=30,
            p_threshold=0.05,
            title='Pancreas L-Trail(scale=30, p<0.05, umap)'
            )

## Comparison with RNA Velocity

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

scv.pl.velocity_embedding_stream(
    adata,
    basis='umap',
    ax=ax1,
    show=False,
    title='scVelo (Stochastic)'
    )

pl.plot_ltrail(
    adata,
    groupby='clusters',
    basis='X_umap',
    use_rep='X_pca',
    method='lmoment',
    scale=30,
    p_threshold=0.05,
    ax=ax2,
    title='L-Trail (umap)',
    legend_loc=None
    )

plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

scv.pl.velocity_embedding_stream(
    adata,
    basis='pca',
    ax=ax1,
    show=False,
    title='scVelo (Stochastic)'
    )

pl.plot_ltrail(
    adata,
    groupby='clusters',
    basis='X_pca',
    use_rep='X_pca',
    method='lmoment',
    scale=30,
    p_threshold=0.05,
    ax=ax2,
    title='L-Trail (PCA)',
    legend_loc=None
    )

plt.tight_layout()
plt.show()

### Advanced Example : Robustness and Limitations in Complex Manifolds

<p align="center">
  <img src="https://raw.githubusercontent.com/jamagiwa/L-Trail/427fe4a0b5491c607f1f0fa2336933fa5f7d2b41/figures/bm_velocity_pca.png" width="48%">
  <img src="https://raw.githubusercontent.com/jamagiwa/L-Trail/c80d3816a0f630845a71dea9d29ac9fea39b9fba/figures/bm_ltrail_vel_pca.png" width="48%">
</p>


In the Bone Marrow dataset, while RNA velocity (left, dynamical model) often infers reversed flows along the Erythroid lineage, L-Trail (right) correctly captures the forward macroscopic progression based on the geometric asymmetry of the cell distribution.

⚠️ **Important Notes on Scope and Limitations**
Before applying L-Trail to your own datasets, please ensure your analysis accounts for the following algorithmic and structural constraints:

* **Geometric Proxy vs. Biological Time:** The inferred directional vectors represent a structural proxy derived from the geometric shape of the data manifold. Unlike RNA velocity, which models a biologically grounded time axis based on transcription and splicing kinetics, L-Trail does not estimate true biological time or physical reaction rates. It strictly captures the macroscopic spatial asymmetry of cell populations.
* **Unidirectional Estimation and the Resolution Trade-off:** Being a cluster-level statistical method, L-Trail infers exactly one macroscopic directional vector per cluster. It cannot intrinsically represent bifurcations or multi-directional branching within a single cell cluster (e.g., the root HSC cluster). While applying a finer clustering resolution can approximate complex branching events by breaking them into smaller segments, this introduces a fundamental trade-off: over-clustering reduces the sample size per cluster, which compromises the statistical stability and accuracy of the L-moment estimation.
* **Sample Size Dependency:** As highlighted by the trade-off above, robust L-moment estimation relies on sufficient sample statistics. While a minimum of 30 cells is often used as a practical threshold, small or highly fragmented clusters may produce unreliable and volatile directional vectors.
* **Linearity Constraint:** Because L-Trail computes directional vectors using linear combinations of moments, the inferred trajectory for a cluster is strictly a linear, straight vector. Consequently, calculations must be performed exclusively within a linear high-dimensional space (e.g., top 30 principal components). Computing vectors directly within non-linear embeddings (e.g., UMAP or t-SNE) introduces severe spatial distortions and is mathematically invalid.
* **Continuous Manifold Assumption:** The geometric inference assumes the presence of an underlying continuous data manifold. The algorithm estimates macroscopic trends within connected, progressing populations and cannot infer developmental trajectories across discrete or disconnected cell states.